In [1]:
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.init as init
import random

In [ ]:
# (do not change this code)
words = open('names.txt').read().splitlines()

In [ ]:
# (do not change this code)
chars = sorted(list(set([c for w in words for c in w])))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

In [ ]:
# (do not change this code)
itos = {i:s for s,i in stoi.items()}

In [ ]:
# Exercise 1: build a simple bigram model for next-character prediction
# - store the co-counts of each character in a 27x27 matrix N
# - compute the normalized probabilities into a matrix P
# - generate a bunch of samples from the model using P and torch.multinomial

In [ ]:
N = torch.zeros(27, 27)  
for w in words:
    w = '.' + w + '.'  
    k = len(w)
    for i in range(k - 1):
        j = i+1
        c1 = w[i]
        c2 = w[j]
        i1 = stoi[c1]  
        i2 = stoi[c2]
        N[i1, i2] += 1  

In [ ]:
# display the matrix
plt.imshow(N)
plt.show()

In [ ]:
#display the matrix in human-readable format
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')
plt.show()

In [ ]:
# useful to reproduce results
g = torch.Generator().manual_seed(2147483647)

In [ ]:
P = N / N.sum(dim=1, keepdim=True)  # Normalize lines
P[torch.isnan(P)] = 0

In [ ]:
# sample 30 words from the model (hint: use torch.multinomial)
for _ in range(30):
    ix = 0
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

In [ ]:
# Exercise 2: build the same bigram model using the NLL loss
# - the dataset is created and encoded for you
# - create the weights matrix W
# - build a training loop to minimize the NLL
# - sample from the model

In [ ]:
# create a training set for bigram model
# (do not change this code) 
xs = []
ys = []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

In [ ]:
import torch.nn.functional as F

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn(27, 27, generator=g)  
W.requires_grad=True

In [ ]:
losses = []
lbd = 100

for k in range(1000):
    # Forward pass
    xenc = F.one_hot(xs, num_classes=27).float() # encode xs with F.one_hot 
    logits = xenc @ W  
    counts = logits.exp()  
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean() 
    # backward pass
    W.grad = None
    loss.backward()
    # update
    W.data -= lbd * W.grad
    print(k, loss.item())

In [ ]:
# finally, sample from the neural net model
g = torch.Generator().manual_seed(2147483647)

for k in range(10):
    out = []
    ix = 0
    for _ in range(10):
        x = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = x @ W  
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
        ix = probs.multinomial(num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

In [ ]:
# Exercise 3: homework (*) extend the previous model to trigram

In [ ]:
# Create training set for trigram model 

xs = []
ys = []
for w in words:
    chs = ['.'] + list(w) + ['.']  
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):  
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append((ix1, ix2))  
        ys.append(ix3)  

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.shape[0]

In [ ]:
# Network initalization 
g = torch.Generator().manual_seed(2147483647)
W = torch.rand(27, 27, 27, generator=g)
W.requires_grad = True

# split 
a = int(.8 * num)
b = int(.9 * num)
xs_train = xs[:a,:]
xs_dev = xs[a:b,:]
xs_test = xs[b:,:]
ys_train = ys[:a]
ys_dev = ys[a:b]
ys_test = ys[b:]
num_train = xs_train.shape[0]
num_dev = xs_dev.shape[0]


In [ ]:
train_loss = []
dev_loss = []
lbd = 1.0 # learning rate

for k in range(1000):
  # forward pass
  xenc = F.one_hot(xs_train, num_classes=27).float()
  logits = torch.einsum('bik,kjl->bil', xenc, W)  # [bs, 2, 27]
  probs = F.softmax(logits, dim=2)
  probs = probs[torch.arange(0, num_train), :, ys_train]
  loss = - torch.mean(torch.log(probs))
  # backward pass
  W.grad = None
  loss.backward()
  # update
  W.data -= lbd * W.grad
  print(k, loss.item())
  train_loss.append(loss.item())

  with torch.no_grad():
    if k % 100 == 0:
      xenc = F.one_hot(xs_dev, num_classes=27).float()
      logits = torch.einsum('bik,kjl->bil', xenc, W)  # [bs, 2, 27]
      probs = F.softmax(logits, dim=2)
      probs = probs[torch.arange(0, num_dev), :, ys_dev]
      loss_dev = - torch.mean(torch.log(probs))
      print(f'\ttrain loss, dev loss = {loss.item()}, {loss_dev.item()}')
      train_loss.append(loss.item())
      dev_loss.append(loss_dev.item())

print("Mean of training loss: ", sum(train_loss)/len(train_loss))
print("Mean of dev set loss: ", sum(dev_loss)/len(dev_loss))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_loss, label='Loss')
plt.title('Loss Evolution')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for k in range(10):
    out = []
    ix_1 = stoi['.']
    ix_2 = torch.randint(0, 26, (1,)).item()
    for _ in range(10):

        x = F.one_hot(torch.tensor([ix_1, ix_2]), num_classes=27).float()  
        logits = torch.einsum('ik,kjl->l', x, W)
        probs = F.softmax(logits, dim=0)
        ix_3 = torch.multinomial(probs, 1).item()

        out.append(itos[ix_2])

        if ix_3 == 0:
            break

        ix_1 = ix_2
        ix_2 = ix_3


    print(''.join(out))

In [ ]:
# Exercise 4: let's build a better model
# Bengio et al. 2003 MLP language model paper, https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# build the dataset (do not change this code)
block_size = 3
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

X_train, Y_train = build_dataset(words[:n1])
X_dev, Y_dev = build_dataset(words[n1:n2])
X_test, Y_test = build_dataset(words[n2:])

In [ ]:
emb_size = 300
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)   
W1 = torch.randn((30, emb_size), generator=g)
b1 = torch.randn(emb_size, generator=g)
W2 = torch.randn((emb_size, 27), generator=g)
b2 = torch.randn((1,27), generator=g)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True  


In [ ]:
# init weights (use torch.nn.init)
b1 = init.zeros_(b1)
b2 = init.zeros_(b2)
W1 = init.uniform_(W1)
W2 = init.uniform_(W2)

In [ ]:
stepi = []
lossi = []

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
# training loop (use the cross-entropy loss)
lbdi = []
lbd = 3
batch_size = 50

for i in range(10000):
    # minibatch
    ix = torch.randint(0, X_train.shape[0], (batch_size,))
    # forward pass
    emb = C[X_train[ix]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[ix])
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    lbd = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data -= lbd * p.grad
    stepi.append(i)
    lossi.append(loss.log10().item())
    lbdi.append(lbd)
    print(i, loss)

In [ ]:
plt.plot(stepi, lossi)
plt.show()

In [ ]:
# compute the train and val loss

In [ ]:
emb = C[X_dev]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_dev)
print(f'val loss = {loss}')

emb = C[X_train]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_train)
print(f'train loss = {loss}')

In [ ]:
# compute the train and val loss.  here are the initial results I got.  
# Improve the loss by playing with the hyper-params and learning rate policy and report the results here.

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 4)

for _ in range(20):
    out = []
    context = [0] * block_size
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))